# 178 — IA para ciencia, clima y salud responsable

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución 1 — VPP a mano

a) Prevalencia 2 % (400 enfermos, 19,600 sanos):
```text
TP = 0.90 × 400   = 360      FP = 0.05 × 19,600 = 980
FN = 40                       TN = 18,620
VPP = 360/1,340 ≈ 26.9 %      VPN = 18,620/18,660 ≈ 99.8 %
```

b) Prevalencia 20 % (4,000 enfermos, 16,000 sanos):
```text
TP = 3,600    FP = 800    VPP = 3,600/4,400 ≈ 81.8 %
```

c) El mismo modelo pasa de VPP 27 % a 82 % sin cambiar un solo peso: el valor
predictivo depende de la prevalencia de la población a la que se aplica. Por eso el
cribado poblacional indiscriminado y el uso sobre población derivada son decisiones
clínicas distintas aunque el modelo sea idéntico.


In [ ]:
N, sens, esp = 20_000, 0.90, 0.95
def vpp(prev):
    enf = N * prev
    tp, fp = sens * enf, (1 - esp) * (N - enf)
    return tp / (tp + fp)
print(round(vpp(0.02), 3), round(vpp(0.20), 3))
assert abs(vpp(0.02) - 0.2687) < 1e-3 and abs(vpp(0.20) - 0.8182) < 1e-3


## Solución 2 — Diagnosticar el estudio

a) **Falta de validación externa / distribution shift**. La caída al cambiar de sitio
es el síntoma clásico (Zech et al. 2018). Corrección: validación multi-sitio
prospectiva y reporte del rendimiento por institución, no del promedio.

b) **Fuga por atajo (shortcut learning)**: el marcador de "portátil" correlaciona con
paciente hospitalizado y por tanto con enfermedad grave. Corrección: eliminar o
enmascarar metadatos visuales, auditar saliencia y probar con equipos distintos.

c) **Métrica equivocada**: con prevalencia 0.5 %, un AUC 0.97 puede dar VPP de un
dígito. Corrección: reportar VPP/VPN a la prevalencia de la población objetivo y el
número de falsos positivos absolutos por cada 10,000 cribados.

d) **Sin efecto en desenlaces**: discriminar bien no es mejorar la salud; si el
sistema no cambia la conducta clínica a tiempo, no hay beneficio. Corrección:
evaluar el sistema sociotécnico completo (alerta + flujo de respuesta) contra
desenlaces, no el modelo aislado.


In [ ]:
diagnosticos = {
    "a": ("falta de validacion externa", "validacion multi-sitio prospectiva"),
    "b": ("fuga por atajo", "enmascarar metadatos y auditar saliencia"),
    "c": ("metrica equivocada", "reportar VPP/VPN a la prevalencia real"),
    "d": ("sin efecto en desenlaces", "ensayo del sistema completo vs desenlaces"),
}
assert len(diagnosticos) == 4 and all(v[0] and v[1] for v in diagnosticos.values())
print("diagnostico completo")


## Solución 3 — El laboratorio como registro

a) Cumplen función de reporte reproducible: la **semilla** (repetibilidad), la lista
**evidence** (hechos inspeccionables) y **limitations** (alcance declarado). Para un
reporte TRIPOD+AI faltarían: descripción de la población y el origen de los datos,
tamaño muestral y su justificación, medidas de calibración además de discriminación,
resultados desagregados por subgrupo, y validación externa independiente.

b) La separación entre lo respaldado y lo no extrapolable es el núcleo de la
honestidad científica porque un resultado sin sus límites declarados invita
exactamente al error que produce los 200 modelos COVID inservibles: leer una métrica
interna como si fuera evidencia de utilidad externa.


In [ ]:
r = run_lab("frontier", seed=178)

def revisar(resultado):
    return {
        "respaldado": list(resultado.get("evidence", [])),
        "no_extrapolable": list(resultado.get("limitations", [])),
    }

rev = revisar(r)
assert rev["respaldado"] and rev["no_extrapolable"]
print(len(rev["respaldado"]), "hechos |", len(rev["no_extrapolable"]), "límites")


## Solución 4 — Meteorología vs clima

a) Confunde **predicción meteorológica** (estado de la atmósfera a días vista,
condicionada a las observaciones actuales, con horizonte de predictibilidad limitado
por el caos) con **proyección climática** (estadística del sistema a décadas bajo
escenarios de forzamiento radiativo). No comparten ni la escala temporal, ni las
variables de interés, ni el criterio de validación.

b) GraphCast aprende la dinámica implícita en 40 años de ERA5, es decir, del clima
**reciente**. Una proyección a 2100 exige simular estados atmosféricos fuera de esa
distribución (mayor forzamiento, regímenes sin análogo histórico), justo donde un
modelo aprendido no ofrece garantías; los modelos climáticos físicos extrapolan
porque codifican leyes de conservación, no correlaciones observadas.

c) Afirmación honesta: "GraphCast supera al modelo determinista operativo HRES del
ECMWF en la mayoría de las variables y plazos evaluados hasta 10 días sobre años
retenidos, con inferencia de ~1 minuto; su desempeño en eventos extremos poco
representados y su uso fuera de la distribución de ERA5 siguen bajo evaluación."
